# Eksponencijalna raspodjela: radni vijek sijalica (knjiga, poglavlje 2.4.4)

$$p(x)=a\,e^{-ax},\qquad P(x)=1-e^{-ax},\qquad x\ge 0\qquad(2.16),(2.17)$$

**Eksperiment.** Knjiga kaže da se eksponencijalnom raspodjelom modeliraju „problemi kašnjenja i radni vijek uređaja“.
Uključimo $N=10\,000$ sijalica. Svaka ispravna sijalica može pregorjeti u svakom satu sa malom vjerovatnoćom $p$,
nezavisno od toga koliko je već radila (sijalica „ne stari“). Mjerimo koliko je sati svaka sijalica radila.

**Zašto eksponencijalna?** Broj sijalica koje pregore u kratkom intervalu proporcionalan je broju onih koje još rade:
$\;\frac{dN(t)}{dt}=-a\,N(t)\;\Rightarrow\;N(t)=N_0e^{-at}$, isto kao kod radioaktivnog raspada.
Udio ispravnih sijalica je vjerovatnoća da sijalica radi duže od $t$:

$$P(T>t)=\frac{N(t)}{N_0}=e^{-at}\quad\Rightarrow\quad P(t)=1-e^{-at},\quad p(t)=a\,e^{-at}.$$

Ista formula slijedi iz $P(T>t)=(1-p)^{t}\approx e^{-pt}$, jer sijalica mora „preživjeti“ svaki od $t$ sati, pa je $a\approx p$.

**Pokretanje:** u Google Colabu `Runtime → Run all`. Prva ćelija sama klonira repozitorij
[ierovic1ba/ssp](https://github.com/ierovic1ba/ssp) (fork repozitorija [esokic/ssp](https://github.com/esokic/ssp) uz knjigu) i učita biblioteku `ssp`.
Svaka bilježnica radi $N=10\,000$ eksperimenata. Detaljna izvođenja su u PDF-u.

In [ ]:
# Priprema okruženja: kloniranje repozitorija sa SSP bibliotekom (prema README uputstvu)
import os, sys, subprocess

REPO_URL = "https://github.com/ierovic1ba/ssp.git"   # fork originala https://github.com/esokic/ssp
SSP_PUTANJA = "/content/ssp" if os.path.isdir("/content") else os.path.abspath("ssp")

if not os.path.isdir(os.path.join(SSP_PUTANJA, "ssp")):
    subprocess.run(["git", "clone", REPO_URL, SSP_PUTANJA], check=True)

sys.path.insert(0, SSP_PUTANJA)   # folder u kojem se nalazi paket ssp/
import ssp

import numpy as np
import matplotlib.pyplot as plt

# Podešavanje prikaza za rad studenata (preporuka iz README-a):
ssp.FORMATIRANJE_PRIKAZA = "JUPYTER"
ssp.REZOLUCIJA = "LOW"

N = 10000   # broj eksperimenata

In [ ]:
p = 0.01             # vjerovatnoća da ispravna sijalica pregori u toku jednog sata
a = p                # parametar eksponencijalne raspodjele [1/h], prosječni vijek 1/a = 100 h

vijek = np.zeros(N)                  # radni vijek svake sijalice [h]
radi = np.ones(N, dtype=bool)        # koje sijalice još rade
broj_ispravnih = [N]
sat = 0
while radi.any():
    sat += 1
    pregorjela = radi & (np.random.uniform(0, 1, N) < p)    # svaka ispravna pregori sa vjerovatnoćom p
    vijek[pregorjela] = sat
    radi[pregorjela] = False
    broj_ispravnih.append(radi.sum())
broj_ispravnih = np.array(broj_ispravnih)

print(f"Posljednja sijalica je pregorjela nakon {sat} sati.")
ssp.slprom1D.realizacije(np.arange(500), vijek[:500],
                         naslov="Radni vijek prvih 500 sijalica [h]")

## Koliko sijalica još radi nakon $t$ sati?
Kriva pada eksponencijalno: nakon $1/a=100$ sati radi još $e^{-1}\approx37\%$ sijalica, nakon 200 sati $e^{-2}\approx13{,}5\%$ itd.

In [ ]:
t = np.arange(len(broj_ispravnih))
ssp.slprom1D.grafik(t, broj_ispravnih, t, N*np.exp(-a*t), stil1="b", stil2="r--",
                    x_labela="vrijeme $t$ [h]", y_labela="broj ispravnih sijalica",
                    naslov="Broj sijalica koje još rade",
                    legenda=["eksperiment", "$N_0 e^{-at}$"])

## Poređenje sa teorijom, sa programskim kodom 2.4 i sa Primjerom 2.13
* **Programski kod 2.4:** argument funkcije `np.random.exponential` je *skala*, tj. srednja vrijednost $1/a$, a ne $a$ iz (2.16).
  Poziv `np.random.exponential(4, N)` zato odgovara $a=1/4$. Ovdje se koristi `np.random.exponential(1/a, N)`.
* **Primjer 2.13 i programski kod 2.41:** ako je $X$ uniformna na $(0,1)$, onda je $Y=-\frac1a\ln(1-X)$ eksponencijalna.
  To je metod inverzne transformacije: $P(Y\le y)=P\big(X\le 1-e^{-ay}\big)=1-e^{-ay}$.

In [ ]:
X_knjiga = np.random.exponential(1/a, N)                     # programski kod 2.4 (skala = 1/a)
Y_inverz = -(1/a) * np.log(1 - np.random.uniform(0, 1, N))  # Primjer 2.13 / programski kod 2.41

x = np.linspace(0, 6/a, 1000)
plt.figure(figsize=(6, 3.5))
plt.hist(vijek, bins=60, range=(0, 6/a), density=True, alpha=0.6, label="sijalice")
plt.hist(X_knjiga, bins=60, range=(0, 6/a), density=True, histtype="step", lw=1.5, color="k", label="np.random.exponential")
plt.hist(Y_inverz, bins=60, range=(0, 6/a), density=True, histtype="step", lw=1.5, color="g", label="$-\\ln(1-X)/a$")
plt.plot(x, a*np.exp(-a*x), "r", lw=2, label="teorija (2.16)")
plt.title(f"Funkcija gustine vjerovatnoće, $a={a}$ 1/h"); plt.xlabel("radni vijek [h]")
plt.legend(); plt.grid(alpha=0.3); plt.show()

xs, Px = ssp.slprom1D.PDF(vijek, step=1, prosirenje_opsega=0.0, prikaz=False)
ssp.slprom1D.grafik(xs, Px, x, 1 - np.exp(-a*x), stil2="r--",
                    x_labela="$t$ [h]", naslov="Funkcija raspodjele vjerovatnoće $P(t)$",
                    legenda=["sijalice", "teorija (2.17)"], lokacija_lgd="lower right")

print(f"E[T]   = {np.mean(vijek):.2f} h     (teorija 1/a   = {1/a:.2f} h)")
print(f"Var[T] = {np.var(vijek):.1f} h^2  (teorija 1/a^2 = {1/a**2:.1f} h^2)")
print(f"medijan = {np.median(vijek):.1f} h   (teorija ln2/a = {np.log(2)/a:.1f} h)")

## Eksponencijalna raspodjela „nema memoriju“
$P(T>s+t\mid T>s)=\dfrac{e^{-a(s+t)}}{e^{-as}}=e^{-at}=P(T>t)$.
Sijalica koja je već radila $s=100$ sati ima **isti** preostali vijek kao nova sijalica, jer se u modelu ne troši.
Kod stvarnih uređaja koji se troše (npr. kočnice) to ne vrijedi, pa za njih eksponencijalna raspodjela nije dobar model.

In [ ]:
s = 100
preostali = vijek[vijek > s] - s                 # preostali vijek sijalica koje su "preživjele" s sati
print(f"sijalica starijih od {s} h: {len(preostali)}")
print(f"prosječni preostali vijek starih sijalica: {np.mean(preostali):.1f} h")
print(f"prosječni vijek novih sijalica:            {np.mean(vijek):.1f} h")

plt.figure(figsize=(6, 3.5))
plt.hist(vijek, bins=50, range=(0, 500), density=True, alpha=0.5, label="nove sijalice")
plt.hist(preostali, bins=50, range=(0, 500), density=True, alpha=0.5, label=f"preostali vijek nakon {s} h")
plt.plot(x, a*np.exp(-a*x), "r", lw=2, label="$a e^{-at}$")
plt.xlim(0, 500); plt.xlabel("[h]"); plt.title("Stara sijalica nije lošija od nove")
plt.legend(); plt.grid(alpha=0.3); plt.show()